In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy comet-ml logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.5 MB/s eta 0:00:00


In [2]:
import sys, os, logging, comet_ml, torch
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

In [3]:
from diploma_code import *

2026-04-14 20:06:30,078 NumExpr defaulting to 4 threads.


In [4]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_cfg = ModelConfig(
    encoder_name="xlm-roberta-base",
    embedding_dim=128,
    query_max_len=256,    # увеличиваем для Phase 2
    doc_max_len=256,
)
model, tokenizer = build_model_and_tokenizer(model_cfg)

ckpt = torch.load("/kaggle/input/models/vikakuznetzowa/checkpoint-phase1-no-eng-best/pytorch/default/1/checkpoint_phase1_no_eng_best.pt",
                   map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel: {n_gpus}x {torch.cuda.get_device_name(0)}")
else:
    print(f"Single GPU: {torch.cuda.get_device_name(0)}")

2026-04-14 20:06:31,812 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-14 20:06:31,814 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-14 20:06:31,895 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

2026-04-14 20:06:32,016 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-14 20:06:32,097 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

2026-04-14 20:06:32,194 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-14 20:06:32,279 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-14 20:06:32,363 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-14 20:06:32,449 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-14 20:06:32,531 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/sentencepiece.bpe.model "HTTP/1.1 200 OK"
2026-04-14 20:06:32,622 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/sentencepiece.bpe.model "HTTP/1.1 200 OK"


...

2026-04-14 20:06:33,551 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-04-14 20:06:33,639 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
2026-04-14 20:06:33,725 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-14 20:06:35,991 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base "HTTP/1.1 307 Temporary Redirect"
2026-04-14 20:06:36,075 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base "HTTP/1.1 200 OK"
2026-04-14 20:06:36,179 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-14 20:06:36,269 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-04-14 20:06:37,438 HTTP Request: HEAD https://huggingf

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DataParallel: 2x Tesla T4


In [ ]:
import json
import random
with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/hard_negatives.json") as f:
    hn_raw = json.load(f)

random.seed(42)
random.shuffle(hn_raw)
MAX_EXAMPLES = 300_000
hn_raw = hn_raw[:MAX_EXAMPLES]


examples = [
    RetrievalExample(query=r["query"], positive=r["positive"],
                     negatives=r["negatives"], lang=r["lang"])
    for r in hn_raw
]
print(f"Loaded {len(examples)} examples with hard negatives")

Loaded 300000 examples with hard negatives


In [ ]:
data_cfg = DataConfig(mixed_lang_ratio=0.0, cache_dir="/kaggle/temp/cache", mmarco_max_triples= 100_000)

train_loader = build_dataloader(
    examples, tokenizer, model_cfg, data_cfg,
    train_cfg_batch_size=48,
    num_workers=2,
    is_train=True,
)

In [8]:
from diploma_code import _load_miracl_examples

val_examples = _load_miracl_examples(data_cfg, split="dev")
val_loader = build_dataloader(
    val_examples, tokenizer, model_cfg, data_cfg,
    train_cfg_batch_size=48, num_workers=0, is_train=False,
)

print(f"Train: {len(examples)} examples, {len(train_loader)} batches/epoch (batch_size=48)")
print(f"Val:   {len(val_examples)} examples, {len(val_loader)} batches")

2026-04-14 20:08:34,642 HTTP Request: HEAD https://huggingface.co/datasets/miracl/miracl/resolve/main/miracl-v1.0-ru/topics/topics.miracl-v1.0-ru-dev.tsv "HTTP/1.1 307 Temporary Redirect"
2026-04-14 20:08:34,654 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/miracl/miracl/5be20db9509754dadad47689368639fcec739c00/miracl-v1.0-ru%2Ftopics%2Ftopics.miracl-v1.0-ru-dev.tsv "HTTP/1.1 200 OK"
2026-04-14 20:08:34,667 HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/miracl/miracl/5be20db9509754dadad47689368639fcec739c00/miracl-v1.0-ru%2Ftopics%2Ftopics.miracl-v1.0-ru-dev.tsv "HTTP/1.1 200 OK"


topics.miracl-v1.0-ru-dev.tsv: 0.00B [00:00, ?B/s]

2026-04-14 20:08:34,766 HTTP Request: HEAD https://huggingface.co/datasets/miracl/miracl/resolve/main/miracl-v1.0-ru/qrels/qrels.miracl-v1.0-ru-dev.tsv "HTTP/1.1 307 Temporary Redirect"
2026-04-14 20:08:34,778 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/miracl/miracl/5be20db9509754dadad47689368639fcec739c00/miracl-v1.0-ru%2Fqrels%2Fqrels.miracl-v1.0-ru-dev.tsv "HTTP/1.1 200 OK"
2026-04-14 20:08:34,791 HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/miracl/miracl/5be20db9509754dadad47689368639fcec739c00/miracl-v1.0-ru%2Fqrels%2Fqrels.miracl-v1.0-ru-dev.tsv "HTTP/1.1 200 OK"


qrels.miracl-v1.0-ru-dev.tsv: 0.00B [00:00, ?B/s]

2026-04-14 20:08:34,821 MIRACL ru dev: 1252 queries, 12607 unique docids to resolve
2026-04-14 20:08:34,943 HTTP Request: HEAD https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ru/docs-0.jsonl.gz "HTTP/1.1 302 Found"
2026-04-14 20:08:35,022 HTTP Request: GET https://huggingface.co/api/datasets/miracl/miracl-corpus/xet-read-token/d921ec7e349ce0d28daf30b2da9da5ee698bef0d "HTTP/1.1 200 OK"


...

2026-04-14 20:10:18,508 HTTP Request: HEAD https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ru/docs-18.jsonl.gz "HTTP/1.1 302 Found"


miracl-corpus-v1.0-ru/docs-18.jsonl.gz:   0%|          | 0.00/81.0M [00:00<?, ?B/s]

2026-04-14 20:10:23,919 HTTP Request: HEAD https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ru/docs-19.jsonl.gz "HTTP/1.1 302 Found"


miracl-corpus-v1.0-ru/docs-19.jsonl.gz:   0%|          | 0.00/6.75M [00:00<?, ?B/s]

Train: 300000 examples, 6250 batches/epoch (batch_size=48)
Val:   1252 examples, 27 batches


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler

train_cfg = TrainConfig(
    output_dir="/kaggle/working/checkpoints",
    per_device_batch_size=48,
    gradient_accumulation_steps=1,
    learning_rate=1.5e-6,
    fp16=True,
    eval_every_n_steps=500,
    save_every_n_steps=0,
    log_every_n_steps=100,
    max_grad_norm=1.0,
)

num_epochs = 1
total_steps = (len(train_loader) // train_cfg.gradient_accumulation_steps) * num_epochs
print(f"OneCycleLR total_steps = {total_steps}  ({total_steps // num_epochs}/epoch, ~{total_steps * 1.5 / 3600:.1f}h estimated)")

optimizer = AdamW(model.parameters(), lr=train_cfg.learning_rate, weight_decay=0.01)
scheduler = OneCycleLR(optimizer, max_lr=train_cfg.learning_rate,
                       total_steps=max(total_steps, 1), pct_start=0.1)
loss_fn = InfoNCELoss(temperature=0.05)
scaler = GradScaler()

OneCycleLR total_steps = 6250  (6250/epoch, ~2.6h estimated)


/tmp/ipykernel_23/567555143.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [10]:
from diploma_code import _init_comet_experiment
comet_cfg = CometConfig(
    enabled=True,
    project_name="colbert-ru",
    experiment_name="ablation-no-mixed-lang-phase-2",
    tags=["ablation", "no-mixed-lang"],
)
experiment = _init_comet_experiment(comet_cfg)
if experiment is not None:
    experiment.log_parameters({
        "phase": "phase2",
        "learning_rate": train_cfg.learning_rate,
        "num_epochs": num_epochs,
        "batch_size": train_cfg.per_device_batch_size,
        "grad_accum": train_cfg.gradient_accumulation_steps,
        "num_train_examples": len(examples),
    })

train_one_phase(
    model, train_loader, val_loader,
    optimizer, scheduler, loss_fn, device,
    train_cfg, num_epochs,
    phase_name="phase2", scaler=scaler,
    comet_experiment=experiment,
)

if experiment is not None:
    experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kuznetzowa-viktoria/colbert-ru/1302b6474850462495f180df5624a22b

COMET INFO: Couldn't find a Git repository in '/kaggle/working' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
/kaggle/usr/lib/notebooks/vikakuznetzowa/diploma_code/diploma_code.py:1627: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/kaggle/usr/lib/notebooks/vikakuznetzowa/diploma_code/diploma_code.py:1640: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch ski